In [2]:
%load_ext autoreload
%autoreload 2

%load_ext sql
%config SqlMagic.autopandas = True
%sql sqlite:///../nba.sqlite

import sqlite3

con = sqlite3.connect("../nba.sqlite")

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'sqlite:///../nba.sqlite'

In [3]:
%%sql
select *
from team


Running query in 'sqlite:///../nba.sqlite'

,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949.0
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946.0
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970.0
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002.0
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966.0
5,1610612742,Dallas Mavericks,DAL,Mavericks,Dallas,Texas,1980.0
6,1610612743,Denver Nuggets,DEN,Nuggets,Denver,Colorado,1976.0
7,1610612744,Golden State Warriors,GSW,Warriors,Golden State,California,1946.0
8,1610612745,Houston Rockets,HOU,Rockets,Houston,Texas,1967.0
9,1610612746,Los Angeles Clippers,LAC,Clippers,Los Angeles,California,1970.0


- Games are uniquely identified by game_id in game table.

In [5]:
%%sql
select *
from game
where 
    (team_id_home = "1610612743" or team_id_away = "1610612743") 
    and season_type = "Regular Season"


Running query in 'sqlite:///../nba.sqlite'

,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,reb_away,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type
0,21977,1610612743,DEN,Denver Nuggets,0027700006,1977-10-19 00:00:00,DEN vs. MIL,W,240,49.0,...,NaN,NaN,NaN,NaN,NaN,32.0,115.0,-18,0,Regular Season
1,21977,1610612743,DEN,Denver Nuggets,0027700016,1977-10-21 00:00:00,DEN vs. LAL,W,240,39.0,...,NaN,NaN,NaN,NaN,NaN,30.0,101.0,-5,0,Regular Season
2,21977,1610612765,DET,Detroit Pistons,0027700028,1977-10-22 00:00:00,DET vs. DEN,W,240,48.0,...,NaN,NaN,NaN,NaN,NaN,29.0,106.0,-20,0,Regular Season
3,21977,1610612754,IND,Indiana Pacers,0027700035,1977-10-25 00:00:00,IND vs. DEN,W,240,52.0,...,NaN,NaN,NaN,NaN,NaN,28.0,104.0,-25,0,Regular Season
4,21977,1610612743,DEN,Denver Nuggets,0027700043,1977-10-26 00:00:00,DEN vs. POR,W,240,39.0,...,NaN,NaN,NaN,NaN,NaN,32.0,108.0,-3,0,Regular Season
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3618,22022,1610612743,DEN,Denver Nuggets,0022201175,2023-04-02 00:00:00,DEN vs. GSW,W,240,44.0,...,57.0,26.0,9.0,4.0,15.0,22.0,110.0,-2,1,Regular Season
3619,22022,1610612745,HOU,Houston Rockets,0022201183,2023-04-04 00:00:00,HOU vs. DEN,W,240,47.0,...,47.0,26.0,9.0,13.0,20.0,18.0,103.0,-21,1,Regular Season
3620,22022,1610612756,PHX,Phoenix Suns,0022201201,2023-04-06 00:00:00,PHX vs. DEN,W,240,41.0,...,38.0,24.0,7.0,2.0,13.0,19.0,115.0,-4,1,Regular Season
3621,22022,1610612762,UTA,Utah Jazz,0022201213,2023-04-08 00:00:00,UTA vs. DEN,W,240,44.0,...,41.0,32.0,7.0,9.0,14.0,20.0,114.0,-4,1,Regular Season


In [17]:
%%sql
DROP TABLE IF EXISTS season;
CREATE TABLE season (
    season_id   TEXT PRIMARY KEY,
    season_name TEXT NOT NULL,
    season_type TEXT NOT NULL
);
INSERT INTO season (season_id, season_name, season_type)
SELECT DISTINCT
    season_id,
    substr(season_id, 2) || '-' || substr(CAST(substr(season_id, 2) AS INTEGER) + 1, 3, 2),
    season_type
FROM game;

Running query in 'sqlite:///../nba.sqlite'

225 rows affected.

""


In [18]:
%%sql
SELECT y.year, t.season_type
FROM (SELECT DISTINCT substr(season_id, 2) AS year FROM season) y
CROSS JOIN (SELECT DISTINCT season_type FROM season) t
LEFT JOIN season s
       ON substr(s.season_id, 2) = y.year AND s.season_type = t.season_type
WHERE s.season_id IS NULL
ORDER BY y.year, t.season_type;

Running query in 'sqlite:///../nba.sqlite'

,year,season_type
0,1946,All-Star
1,1946,Pre Season
2,1947,All-Star
3,1947,Pre Season
4,1948,All-Star
...,...,...
78,2004,Pre Season
79,2005,Playoffs
80,2012,Regular Season
81,2018,Pre Season
